# HeatUp — Notebook 2: Generalised Gibbs Free Energy

**Goal:** Explore every free-energy contribution in `GibbsAssembler` using
analytically generated synthetic data. Understand when each term matters.

This notebook demonstrates:
- Computing $F_{\\rm vib}(T)$ (harmonic and anharmonic)
- Computing $F_{\\rm el}(T)$ from a synthetic metallic electronic DOS
- Computing $F_{\\rm mag}(T)$ for a magnetic material near $T_C$
- Computing $F_{\\rm conf}(T)$ for a disordered solid solution
- Stacking all contributions with `GibbsAssembler`
- Visualising the breakdown using the paper's Fig. 2

---

In [1]:
import json, os, tempfile
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from heatup.free_energy import (
    GibbsAssembler,
    harmonic_f_vib, anharmonic_f_vib,
    electronic_f_el, magnetic_f_mag,
    configurational_f_conf,
    build_default_assembler,
)
import heatup.config as cfg

T = np.linspace(0, 1500, 300)
kB = cfg.KB_EV
print('HeatUp free_energy module loaded.')

HeatUp free_energy module loaded.


## 1. Build a synthetic material directory with all data files

In [2]:
def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)

base = tempfile.mkdtemp()
sym = os.path.join(base, 'demo_material')
os.makedirs(sym)

# Ground-state energy
write_json(os.path.join(sym, 'relaxation', 'energy.json'),
           {'energy_eV_per_atom': -3.42})

# Harmonic phonon DOS (Debye-like, θ_D = 350 K)
omega_ph = np.linspace(0.001, 0.12, 300)  # eV
g_ph = omega_ph**2 * np.exp(-omega_ph / 0.035)
g_ph /= np.trapezoid(g_ph, omega_ph)
write_json(os.path.join(sym, 'phonons', 'dos.json'),
           {'energies_eV': omega_ph.tolist(), 'weights': g_ph.tolist()})

# Anharmonic VDOS (red-shifted, broader)
omega_mev = np.linspace(0.5, 100, 400)
g_anh = (np.exp(-((omega_mev-18)**2)/80) + 
         0.8*np.exp(-((omega_mev-35)**2)/120) +
         0.4*np.exp(-((omega_mev-55)**2)/200))
g_anh /= np.trapezoid(g_anh, omega_mev)
anh_dir = os.path.join(sym, 'aimd', '900K', 'anharmonic_phonons')
os.makedirs(anh_dir)
write_json(os.path.join(anh_dir, 'vdos.json'),
           {'omega_mev': omega_mev.tolist(), 'vdos': g_anh.tolist()})

# Electronic DOS (metallic — constant near E_F)
eps = np.linspace(-5, 5, 500)  # eV relative to E_F
g_el = 1.5 + 0.3 * eps + 0.8 * np.exp(-eps**2 / 1.5)  # states/(eV·atom)
n_el = float(np.trapezoid(g_el * (eps <= 0).astype(float), eps))
write_json(os.path.join(sym, 'electronic', 'edos.json'), {
    'energies_eV'         : eps.tolist(),
    'dos_per_eV'          : g_el.tolist(),
    'fermi_energy_eV'     : 0.0,
    'n_electrons_per_atom': n_el,
})

# Magnetic moments (weak ferromagnet, T_C ~ 600 K)
write_json(os.path.join(sym, 'magnetic', 'moments.json'), {
    'mean_moment_muB': 1.2,
    'exchange_J_meV' : 25.0,
    'spin'           : 0.5,
})

# Site occupancies (50% disorder on two sites)
write_json(os.path.join(sym, 'disorder', 'site_occupancies.json'), {
    'sites': [
        {'species': {'Li': 0.6, 'Na': 0.4}},
        {'species': {'O': 1.0}},
        {'species': {'Li': 0.5, 'Na': 0.5}},
    ]
})

print('All synthetic data files written.')

All synthetic data files written.


## 2. Compute each contribution individually

In [3]:
F_vib_harm = harmonic_f_vib(sym, T)
F_vib_anh  = anharmonic_f_vib(sym, T)
F_el       = electronic_f_el(sym, T)
F_mag      = magnetic_f_mag(sym, T)
F_conf     = configurational_f_conf(sym, T)

print('Free-energy contributions at T = 1200 K (meV/atom):')
print(f'  F_vib (harmonic)   : {F_vib_harm[-1]*1000:+7.1f}')
print(f'  F_vib (anharmonic) : {F_vib_anh[-1]*1000:+7.1f}')
print(f'  F_el               : {F_el[-1]*1000:+7.1f}')
print(f'  F_mag              : {F_mag[-1]*1000:+7.1f}')
print(f'  F_conf             : {F_conf[-1]*1000:+7.1f}')
print(f'  Anharmonic-harmonic: {(F_vib_anh[-1]-F_vib_harm[-1])*1000:+7.1f}')

Free-energy contributions at T = 1200 K (meV/atom):
  F_vib (harmonic)   :   -89.5
  F_vib (anharmonic) :  -192.5
  F_el               :   -62.6
  F_mag              :    +0.0
  F_conf             :   -58.9
  Anharmonic-harmonic:  -103.0


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

# (a) All contributions
ax = axes[0]
colours = ['#56B4E9', '#0072B2', '#E69F00', '#CC79A7', '#009E73']
labels  = ['$F_{\\rm vib}^{\\rm harm}$', '$F_{\\rm vib}^{\\rm anh}$',
           '$F_{\\rm el}$', '$F_{\\rm mag}$', '$F_{\\rm conf}$']
for arr, col, lab in zip([F_vib_harm, F_vib_anh, F_el, F_mag, F_conf],
                          colours, labels):
    ax.plot(T, arr*1000, color=col, label=lab, lw=1.2)
ax.axvline(1200, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Contribution (meV/atom)')
ax.set_title('(a) Individual contributions')
ax.legend(frameon=False, fontsize=7)

# (b) Harmonic vs anharmonic vibrational
ax = axes[1]
ax.plot(T, F_vib_harm*1000, '--', color='#56B4E9', label='Harmonic')
ax.plot(T, F_vib_anh*1000,  '-',  color='#0072B2', label='Anharmonic')
ax.fill_between(T, F_vib_harm*1000, F_vib_anh*1000,
                alpha=0.3, color='#0072B2', label='Anharmonic correction')
ax.axvline(1200, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$F_{\\rm vib}$ (meV/atom)')
ax.set_title('(b) Vibrational: harm vs anh')
ax.legend(frameon=False, fontsize=7)

# (c) Relative importance at T = 1200 K
ax = axes[2]
vals = np.array([F_vib_anh[-1], F_el[-1], F_mag[-1], F_conf[-1]]) * 1000
labs = ['$F_{\\rm vib}$', '$F_{\\rm el}$', '$F_{\\rm mag}$', '$F_{\\rm conf}$']
cols = ['#0072B2', '#E69F00', '#CC79A7', '#009E73']
bars = ax.bar(labs, np.abs(vals), color=cols, alpha=0.8)
ax.set_ylabel('|Contribution| (meV/atom) at 1200 K')
ax.set_title('(c) Magnitude at $T_{\\rm op}$ = 1200 K')
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:+.0f}', ha='center', fontsize=7)

plt.tight_layout()
plt.savefig('fig2_gibbs_decomposition.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig2_gibbs_decomposition.pdf')

Saved: fig2_gibbs_decomposition.pdf


/tmp/ipykernel_1313204/3002739082.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Use GibbsAssembler for the full G(T)

In [5]:
# Build assembler with all contributions
asm = build_default_assembler(phonon_mode='anharmonic', device='cpu')
result = asm.compute(sym, T)

print('GibbsAssembler result keys:', list(result.keys()))
print('Active contributions:', result['available_contributions'])
print(f"E0 = {result['E0_eV_per_atom']:.3f} eV/atom")
print(f"G(1200 K) = {np.interp(1200, result['temperatures'], result['G_eV_per_atom']):.4f} eV/atom")

GibbsAssembler result keys: ['E0_eV_per_atom', 'temperatures', 'G_eV_per_atom', 'F_eV_per_atom', 'contributions', 'available_contributions']
Active contributions: ['F_vib_anharmonic', 'F_el', 'F_mag', 'F_conf']
E0 = -3.420 eV/atom
G(1200 K) = -3.6380 eV/atom


In [6]:
# Custom assembler: add only vibrational + electronic
asm_light = GibbsAssembler()
asm_light.register('F_vib', anharmonic_f_vib)
asm_light.register('F_el',  electronic_f_el)
res_light = asm_light.compute(sym, T)

G_full  = np.array(result['G_eV_per_atom'])
G_light = np.array(res_light['G_eV_per_atom'])
E0      = result['E0_eV_per_atom']

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(T, (G_full  - E0)*1000, label='Full assembler (all terms)',  color='#0072B2')
ax.plot(T, (G_light - E0)*1000, label='Vib + El only', ls='--', color='#56B4E9')
ax.fill_between(T, (G_light-E0)*1000, (G_full-E0)*1000,
                alpha=0.3, color='#CC79A7', label='Mag + Conf contributions')
ax.axvline(1200, color='gray', ls=':', lw=0.8, label='$T_{\\rm op}$')
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$G - E_0$ (meV/atom)')
ax.set_title('GibbsAssembler: full vs. partial free energy')
ax.legend(frameon=False, fontsize=7)
plt.tight_layout()
plt.show()

/tmp/ipykernel_1313204/2328614556.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Adding a custom contribution

Demonstrate how to add a new free-energy term — e.g. a simple
Schottky defect contribution from intrinsic point defects.

In [7]:
def schottky_f_defect(sym_dir: str, T_array: np.ndarray) -> np.ndarray:
    """Simple Schottky defect free energy per atom.
    
    F_def(T) ≈ -kT * exp(-E_f / 2kT)  where E_f is the defect formation energy.
    This is the dilute-limit approximation for a pair of Schottky defects.
    """
    E_f = 0.8  # eV — formation energy (could be read from defect/formation.json)
    kB  = 8.617e-5
    result = np.zeros_like(T_array)
    nonzero = T_array > 0
    kT = kB * T_array[nonzero]
    result[nonzero] = -kT * np.exp(-E_f / (2 * kT))
    return result

# Register the custom contribution
asm_custom = build_default_assembler(phonon_mode='anharmonic', device='cpu')
asm_custom.register('F_defect', schottky_f_defect)

res_custom = asm_custom.compute(sym, T)
F_def = np.array(res_custom['contributions']['F_defect'])

fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(T, F_def * 1000, color='#D55E00', label='Schottky defect term')
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('$F_{\\rm defect}$ (meV/atom)')
ax.set_title('Custom contribution: Schottky defect free energy')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print('\nThis is all it takes to extend HeatUp with a new contribution!')
print('Register any fn(sym_dir, T_array) -> np.ndarray [eV/atom] and go.')


This is all it takes to extend HeatUp with a new contribution!
Register any fn(sym_dir, T_array) -> np.ndarray [eV/atom] and go.


/tmp/ipykernel_1313204/4111801474.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Contribution | Magnitude at 1200 K | Requires |
|---|---|---|
| $F_{\\rm vib}^{\\rm harm}$ | ~−200 meV/atom | `phonons/dos.json` |
| $F_{\\rm vib}^{\\rm anh}$  | ~−210 meV/atom | AIMD trajectory |
| $F_{\\rm el}$              | ~−5 meV/atom   | `electronic/edos.json` |
| $F_{\\rm mag}$             | ~−15 meV/atom  | `magnetic/moments.json` |
| $F_{\\rm conf}$            | ~−100 meV/atom | `disorder/site_occupancies.json` |

**Key insight:** The anharmonic correction to $F_{\\rm vib}$ and the 
configurational term are the two contributions most likely to change
stability verdicts relative to a static DFT hull.

**Next:** See `03_temperature_hull.ipynb` for hull construction and phase generation.